# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [ ]:
# %pip install mlflow

In [2]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [3]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

2026/05/25 21:31:20 INFO mlflow.tracking.fluent: Experiment with name 'check-localhost-connection' does not exist. Creating a new experiment.


🏃 View run abrasive-fawn-701 at: http://localhost:5000/#/experiments/1/runs/74c84d04481a40d3bebeaae75d12e508
🧪 View experiment at: http://localhost:5000/#/experiments/1


Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [4]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2026/05/25 21:31:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 21:31:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'sk-learn-random-forest-reg-model'.
2026/05/25 21:31:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 1
Created version '1' of model 'sk-learn-random-forest-reg-model'.


🏃 View run burly-newt-475 at: http://localhost:5000/#/experiments/1/runs/865247d716b740daa89d36a6db51777f
🧪 View experiment at: http://localhost:5000/#/experiments/1


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [5]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.49980732  44.5541425   41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.44675489 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [6]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [11]:
# %pip install hyperopt
%pip install -U git+https://github.com/hyperopt/hyperopt

  Cloning https://github.com/hyperopt/hyperopt to /tmp/pip-req-build-012owv37
  Running command git clone --filter=blob:none --quiet https://github.com/hyperopt/hyperopt /tmp/pip-req-build-012owv37
  Resolved https://github.com/hyperopt/hyperopt to commit c49ad148201c81c6ad1b43730ef4d0912734aa37
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hyperopt: filename=hyperopt-0.3.0-py3-none-any.whl size=970867 sha256=09874071767ad8577369dd68e0ffbb4a99308c94710ed6aaed302a8112a75071
  Stored in directory: /tmp/pip-ephem-wheel-cache-4r2ilz98/wheels/ad/e0/dc/af4d21315718e63bc2e53ded682ca11817b02cb72d3935cf0d
Successfully built hyperopt
  Attempting uninstall: hyperopt
    Found existing installation: hyperopt 0.2.7
    Uninstalling hyperopt-0.2.7:
      Successfully uninstalled hyperopt-0.2.7
Note: you may need to restart the kernel to use updated packages.


In [12]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [13]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [14]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

2026/05/25 21:37:54 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779737874699, experiment_id='2', last_update_time=1779737874699, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>

In [15]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

  0%|          | 0/8 [00:00<?, ?trial/s, best loss=?]

I0000 00:00:1779737880.255791   98883 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3979 MB memory:  -> device: 0, name: NVIDIA RTX A3000 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/3                                            

  0%|          | 0/8 [00:03<?, ?trial/s, best loss=?]

I0000 00:00:1779737883.245442  112507 service.cc:153] XLA service 0x7dddb802e600 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779737883.245468  112507 service.cc:161]   StreamExecutor [0]: NVIDIA RTX A3000 Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.1)
I0000 00:00:1779737883.285394  112507 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779737883.415285  112507 cuda_dnn.cc:461] Loaded cuDNN version 92101
I0000 00:00:1779737883.434788  112507 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_659__.8


 1/46 ━━━━━━━━━━━━━━━━━━━━ 2:15 3s/step - loss: 42.8543 - root_mean_squared_error: 6.5463
12/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 23.4503 - root_mean_squared_error: 4.7490 
22/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 17.1006 - root_mean_squared_error: 3.9867
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 13.1301 - root_mean_squared_error: 3.4453
  0%|          | 0/8 [00:06<?, ?trial/s, best loss=?]

...

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 20.5895 - root_mean_squared_error: 4.4831
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - loss: 12.3306 - root_mean_squared_error: 3.5115 - val_loss: 3.1506 - val_root_mean_squared_error: 1.7750

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - loss: 3.7664 - root_mean_squared_error: 1.9407
 9/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.2570 - root_mean_squared_error: 1.8032 
18/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.0149 - root_mean_squared_error: 1.7341
28/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.8477 - root_mean_squared_error: 1.6847
39/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7397 - root_mean_squared_error: 1.6523
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.6971 - root_mean_squared_error: 1.6394
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 2.3855 - root_mean_squared_error: 1.5445 - val_loss: 2.0318 - val_root_mean_squared_error: 1.4254

Epoc

2026/05/25 21:40:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run flawless-duck-137 at: http://localhost:5000/#/experiments/2/runs/8a0c592c9689444c9d20da081719d037

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [02:38<00:00, 19.80s/trial, best loss: 0.7731749415397644]


2026/05/25 21:40:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best parameters: {'lr': 0.023388641607125372, 'momentum': 0.6243091106158793}
Best eval rmse: 0.7731749415397644
🏃 View run youthful-hare-83 at: http://localhost:5000/#/experiments/2/runs/174f1362a2bf4eeabace76d858abeabb
🧪 View experiment at: http://localhost:5000/#/experiments/2


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [16]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

  0%|          | 0/8 [00:00<?, ?trial/s, best loss=?]

I0000 00:00:1779738475.111455  112510 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_15024__.8


 1/46 ━━━━━━━━━━━━━━━━━━━━ 1:16 2s/step - loss: 39.6313 - root_mean_squared_error: 6.2953
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 14.3147 - root_mean_squared_error: 3.6160 
23/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 10.2411 - root_mean_squared_error: 3.0090
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.3847 - root_mean_squared_error: 2.6991 
40/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 7.3208 - root_mean_squared_error: 2.5094
  0%|          | 0/8 [00:02<?, ?trial/s, best loss=?]

I0000 00:00:1779738476.263744  112510 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_15024__.8


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 6.7183 - root_mean_squared_error: 2.3964
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - loss: 2.5932 - root_mean_squared_error: 1.6103 - val_loss: 0.8373 - val_root_mean_squared_error: 0.9150

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - loss: 1.0973 - root_mean_squared_error: 1.0475
 7/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.8040 - root_mean_squared_error: 0.8934 
15/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7264 - root_mean_squared_error: 0.8497
21/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7101 - root_mean_squared_error: 0.8407
30/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7059 - root_mean_squared_error: 0.8388
40/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7019 - root_mean_squared_error: 0.8367
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.6569 - root_mean_squared_error: 0.8105 - val_loss: 0.6222 - val_root_mean_squared_error: 0.7888

Epoch 3/3                       

...

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 32.4889 - root_mean_squared_error: 5.6999
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 32.2805 - root_mean_squared_error: 5.6816 - val_loss: 31.5777 - val_root_mean_squared_error: 5.6194

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - loss: 30.7498 - root_mean_squared_error: 5.5453
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 31.8341 - root_mean_squared_error: 5.6420 
24/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 31.5030 - root_mean_squared_error: 5.6126
37/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 31.3212 - root_mean_squared_error: 5.5964
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 30.9604 - root_mean_squared_error: 5.5642 - val_loss: 30.2730 - val_root_mean_squared_error: 5.5021

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - loss: 29.5136 - root_mean_squared_error: 5.4326
16/

2026/05/25 21:50:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run victorious-mare-731 at: http://localhost:5000/#/experiments/2/runs/7192d70eedf345ea8ccb7bad9ef1d463

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [02:50<00:00, 21.36s/trial, best loss: 0.7468439340591431]


2026/05/25 21:50:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best parameters: {'lr': 0.03473212466065397, 'momentum': 0.4482262162232161}
Best eval rmse: 0.7468439340591431
🏃 View run bedecked-bat-119 at: http://localhost:5000/#/experiments/2/runs/1b067d98de9c4c13a09b1bd050458288
🧪 View experiment at: http://localhost:5000/#/experiments/2


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [17]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

ConnectionError: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5002): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
response.content